# cr-train notebook, but easier to read

This follows `main.py`, but makes the end result nicer to inspect in Colab.

- bootstrap with `uv`
- set a few knobs
- plug in model pieces
- preview one batch
- collect flat history rows
- show a clean table, a clean plot, and a compact test summary


## Start here

This keeps setup light. It uses `uv`, installs into the current notebook kernel, and picks a checkpoint folder automatically.

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
WORKDIR = Path.cwd()


def ensure_uv() -> None:
    try:
        subprocess.run(
            ["uv", "--version"],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)


def install_with_uv() -> None:
    # Installing into the current kernel keeps notebook imports simple.
    if (WORKDIR / "pyproject.toml").exists():
        subprocess.run(["uv", "pip", "install", "--system", "-e", "."], check=True)
    else:
        subprocess.run(
            ["uv", "pip", "install", "--system", "git+https://github.com/smturtle2/cr-train.git"],
            check=True,
        )


ensure_uv()
install_with_uv()

# Optional in Colab:
# from google.colab import drive
# drive.mount("/content/drive")

if IN_COLAB and Path("/content/drive/MyDrive").exists():
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/cr-project/checkpoints")
else:
    CHECKPOINT_DIR = Path("/content/checkpoints") if IN_COLAB else WORKDIR / "checkpoints"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"IN_COLAB={IN_COLAB}")
print(f"WORKDIR={WORKDIR}")
print(f"CHECKPOINT_DIR={CHECKPOINT_DIR}")

## Imports

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn

from cr_train import MAE, Trainer, TrainerConfig, build_sen12mscr_loaders

## Quick knobs

Start small first. A few batches are enough to confirm the full path is wired correctly.

In [ ]:
batch_size = 4
seed = 42
split = "official"
max_epochs = 1
train_max_batches = 8
val_max_batches = 2
test_max_batches = 2
resume = None
run_test = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

## Your hooks

Replace these with your project code. The helpers below flatten the trainer records and keep the final plots tidy.

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_model() -> nn.Module:
    # Trainer calls model(sar, cloudy).
    raise NotImplementedError("Replace build_model() with your project model.")


def build_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    # Keep this aligned with build_model().
    raise NotImplementedError("Replace build_optimizer() with your optimizer.")


def build_criterion() -> nn.Module:
    return nn.L1Loss()


def build_metrics() -> list[nn.Module]:
    return [MAE()]


def flatten_record(record: dict) -> dict[str, int | float]:
    row = {
        "epoch": int(record["epoch"]),
        "global_step": int(record["global_step"]),
    }
    for stage in ("train", "val", "test"):
        for name, value in record.get(stage, {}).items():
            row[f"{stage}_{name}"] = float(value)
    return row


def format_row(row: dict[str, int | float]) -> str:
    metric_keys = sorted(key for key in row if key not in {"epoch", "global_step"})
    parts = [f"epoch={row['epoch']}", f"global_step={row['global_step']}"]
    parts.extend(f"{key}={row[key]:.4f}" for key in metric_keys)
    return " | ".join(parts)


def style_frame(frame: pd.DataFrame, caption: str):
    metric_cols = [col for col in frame.columns if col not in {"epoch", "global_step"}]
    style = frame.style.hide(axis="index").set_caption(caption)
    if metric_cols:
        style = style.format({col: "{:.4f}" for col in metric_cols})
    return style


def plot_history_df(history_df: pd.DataFrame) -> None:
    metric_cols = [col for col in history_df.columns if col not in {"epoch", "global_step"}]
    if not metric_cols:
        return

    loss_cols = [col for col in metric_cols if col.endswith("_loss")]
    other_cols = [col for col in metric_cols if col not in loss_cols]
    panels = []
    if loss_cols:
        panels.append(("loss", loss_cols))
    if other_cols:
        panels.append(("metrics", other_cols))

    fig, axes = plt.subplots(
        len(panels),
        1,
        figsize=(10, 4 * len(panels)),
        sharex=True,
        constrained_layout=True,
    )
    if len(panels) == 1:
        axes = [axes]

    colors = plt.get_cmap("tab10")
    epochs = history_df["epoch"]

    for ax, (title, cols) in zip(axes, panels):
        for index, col in enumerate(cols):
            ax.plot(
                epochs,
                history_df[col],
                label=col.replace("_", " "),
                color=colors(index % 10),
                linewidth=2.4,
                marker="o",
                markersize=5,
            )
        ax.set_title(title)
        ax.set_ylabel("value")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False, ncol=min(3, len(cols)))
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[-1].set_xlabel("epoch")
    fig.suptitle("training history", fontsize=14)
    plt.show()


def show_history(history: list[dict[str, int | float]]) -> pd.DataFrame | None:
    if not history:
        print("No new epochs ran.")
        return None

    history_df = pd.DataFrame(history)
    display(style_frame(history_df, "history"))
    plot_history_df(history_df)

    for best_key in ("val_loss", "train_loss"):
        if best_key in history_df.columns:
            best_row = history_df.loc[[history_df[best_key].idxmin()]]
            display(style_frame(best_row, f"best row by {best_key}"))
            break

    return history_df


def show_test_metrics(metrics: dict[str, float]) -> pd.DataFrame | None:
    if not metrics:
        return None

    test_df = pd.DataFrame([{f"test_{name}": float(value) for name, value in metrics.items()}])
    display(style_frame(test_df, "test metrics"))

    series = test_df.iloc[0].sort_values()
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.7 * len(series))))
    ax.barh(series.index, series.values, color="#4C78A8")
    ax.set_title("test metrics")
    ax.set_xlabel("value")
    ax.grid(True, axis="x", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for value, label in zip(series.values, series.index):
        ax.text(value, label, f" {value:.4f}", va="center")
    plt.show()
    return test_df

## Peek at a batch

This is the nice notebook-only part. It lets you check tensor shapes before touching the model.

In [ ]:
seed_everything(seed)

train_loader, val_loader, test_loader = build_sen12mscr_loaders(
    batch_size=batch_size,
    seed=seed,
    split=split,
)

# Taking one batch here is fine. Trainer will create a fresh iterator later.
sample_batch = next(iter(train_loader))
sar, cloudy = sample_batch["inputs"]
target = sample_batch["target"]

print("sar   :", tuple(sar.shape), sar.dtype)
print("cloudy:", tuple(cloudy.shape), cloudy.dtype)
print("target:", tuple(target.shape), target.dtype)
print("metadata keys:", list(sample_batch["metadata"].keys()))

## Build the trainer

In [ ]:
model = build_model().to(device)
optimizer = build_optimizer(model)
criterion = build_criterion()
metrics = build_metrics()

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    metrics=metrics,
    config=TrainerConfig(
        max_epochs=max_epochs,
        train_max_batches=train_max_batches,
        val_max_batches=val_max_batches,
        test_max_batches=test_max_batches,
        checkpoint_dir=CHECKPOINT_DIR,
    ),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
)

if resume is not None:
    trainer.load_checkpoint(resume)

trainer

## Run it

In [ ]:
history = []

for record in trainer.step():
    row = flatten_record(record)
    history.append(row)
    print(format_row(row))

history_df = show_history(history)

if run_test:
    test_metrics = trainer.test()
    test_df = show_test_metrics(test_metrics)
else:
    test_metrics = None
    test_df = None